# Run summary (last task of the daily job)

Runs even when an earlier task failed (`run_if: ALL_DONE`). Collects row counts per table in Bronze / Silver / Gold, rows added today, and the top 10 of `agg_restaurant_ranking`, then returns them as JSON with `dbutils.notebook.exit`, so a chat session can review a run through the Jobs API (`runs/get-output`) without SQL access.

In [ ]:
# --- Config ---
for layer in ("bronze", "silver", "gold"):
    dbutils.widgets.text(layer, f"prod.{layer}")
LAYERS = {layer: dbutils.widgets.get(layer) for layer in ("bronze", "silver", "gold")}


In [ ]:
# --- Summary ---
import json
from datetime import datetime, timezone, timedelta

today = (datetime.now(timezone.utc) + timedelta(hours=7)).date().isoformat()  # VN date
summary = {"today_vn": today, "tables": {}, "errors": []}

for layer, schema in LAYERS.items():
    for row in spark.sql(f"SHOW TABLES IN {schema}").collect():
        if row.isTemporary:
            continue
        name = f"{schema}.{row.tableName}"
        try:
            cols = set(spark.table(name).columns)
            today_col = next((c for c in ("ingest_date", "snapshot_date", "etl_date") if c in cols), None)
            extra = f", COUNT_IF(CAST({today_col} AS DATE) = DATE'{today}') AS today" if today_col else ""
            r = spark.sql(f"SELECT COUNT(*) AS n{extra} FROM {name}").collect()[0].asDict()
            summary["tables"][f"{layer}.{row.tableName}"] = r
        except Exception as e:
            summary["errors"].append(f"{name}: {str(e)[:200]}")

agg = f"{LAYERS['gold']}.agg_restaurant_ranking"
try:
    df = spark.table(agg)
    order = "legit_score DESC NULLS LAST" if "legit_score" in df.columns else "ranking_score DESC"
    keep = [c for c in ("restaurant_name", "district", "city", "video_count", "credited_views",
                        "google_rating", "legit_score", "legit_label", "ranking_score") if c in df.columns]
    summary["top10"] = [r.asDict() for r in spark.sql(f"SELECT {', '.join(keep)} FROM {agg} ORDER BY {order} LIMIT 10").collect()]
except Exception as e:
    summary["errors"].append(f"{agg}: {str(e)[:200]}")

# Dashboard review: how the restaurant_finder filters and sort split the data
try:
    cols = set(spark.table(agg).columns)
    if "legit_score" in cols:
        district = ("CASE WHEN district RLIKE '(?iu)^(q\\\\.?\\\\s*|quận\\\\s+)0*[0-9]+$' "
                    "THEN concat('Quận ', CAST(regexp_extract(district, '([0-9]+)', 1) AS INT)) "
                    "ELSE COALESCE(district, 'Chưa rõ') END")
        def counts(expr, limit=25):
            rows = spark.sql(f"SELECT {expr} AS k, COUNT(*) AS n FROM {agg} GROUP BY 1 ORDER BY n DESC LIMIT {limit}").collect()
            return {str(r.k): r.n for r in rows}
        summary["dashboard"] = {
            "coverage": spark.sql(f"""
                SELECT COUNT(*) AS restaurants,
                       COUNT_IF(latitude IS NOT NULL) AS with_location,
                       COUNT_IF(google_rating IS NOT NULL) AS with_google_rating,
                       COUNT_IF(price_band <> 'Chưa rõ') AS with_price,
                       COUNT_IF(cuisine <> 'Chưa rõ') AS with_cuisine,
                       COUNT_IF(district IS NOT NULL) AS with_district,
                       COUNT_IF(channel_count >= 2) AS with_2plus_channels,
                       COUNT_IF(video_count >= 2) AS with_2plus_videos
                FROM {agg}""").collect()[0].asDict(),
            "legit_label": counts("legit_label"),
            "cuisine": counts("cuisine"),
            "price_band": counts("price_band"),
            "city": counts("city"),
            "district": counts(district),
            "legit_score_buckets": counts("CAST(floor(legit_score / 10) * 10 AS INT)"),
            "top10_hot": [r.asDict() for r in spark.sql(
                f"SELECT restaurant_name, district, cuisine, google_rating, google_rating_count, channel_count, "
                f"legit_score, legit_label, round(100 * ranking_score, 1) AS hot_score "
                f"FROM {agg} ORDER BY ranking_score DESC LIMIT 10").collect()],
        }
except Exception as e:
    summary["errors"].append(f"dashboard stats: {str(e)[:200]}")

# dim_restaurant versions: why are there more versions than current restaurants?
try:
    dim = f"{LAYERS['gold']}.dim_restaurant"
    summary["dim_restaurant_versions"] = spark.sql(f"""
        WITH ids AS (
            SELECT restaurant_id, COUNT(*) AS versions, COUNT_IF(is_current) AS cur,
                   MAX(CASE WHEN restaurant_id LIKE 'place:%' THEN 1 ELSE 0 END) AS is_place
            FROM {dim} WHERE restaurant_key <> -1 GROUP BY restaurant_id)
        SELECT COUNT(*) AS restaurant_ids,
               COUNT_IF(cur = 1) AS with_current,
               COUNT_IF(cur = 0) AS closed_left_source,
               COUNT_IF(cur = 0 AND is_place = 0) AS closed_name_district_ids,
               COUNT_IF(versions > 1) AS ids_with_several_versions,
               MAX(versions) AS max_versions,
               (SELECT COUNT(DISTINCT valid_from) FROM {dim} WHERE restaurant_key <> -1) AS distinct_runs_with_new_versions
        FROM ids""").collect()[0].asDict()
except Exception as e:
    summary["errors"].append(f"dim_restaurant versions: {str(e)[:200]}")

out = json.dumps(summary, ensure_ascii=False, default=str)
print(out)
dbutils.notebook.exit(out[:900_000])
